# Cấu hình

In [63]:
REPO_URL = "https://github.com/thieuphong220203/vietnamese-voice-qa.git"
BRANCH = "main"
PROJECT_DIR = "/content/AdvancedML"

# Không cache model trên Google Drive vì file model đã bị lưu sai thành 79 bytes.
USE_DRIVE_CACHE = False

# Audio dùng để test ASR.
SAMPLE_AUDIO = "data/samples/fleurs_vi/audio/sample_000.wav"

# Mount Drive và cấu hình model cache

In [64]:
import os

if USE_DRIVE_CACHE:
    raise RuntimeError(
        "Không nên dùng Drive cache cho PhoWhisper vì file trọng số "
        "có thể bị lưu thành file con trỏ 79 bytes."
    )

os.environ["HF_HOME"] = "/content/huggingface"
os.environ["HF_HUB_CACHE"] = "/content/huggingface/hub"
os.environ["HF_HUB_DISABLE_XET"] = "1"

print("HF_HOME:", os.environ["HF_HOME"])
print("HF_HUB_CACHE:", os.environ["HF_HUB_CACHE"])

HF_HOME: /content/huggingface
HF_HUB_CACHE: /content/huggingface/hub


# Update code

In [65]:
import os
import subprocess
from pathlib import Path

project_path = Path(PROJECT_DIR)
git_path = project_path / ".git"


def git(*args: str) -> None:
    subprocess.run(
        ["git", "-C", str(project_path), *args],
        check=True,
    )


if git_path.is_dir():
    print("Repository đã tồn tại, đang cập nhật...")

    # Repo cũ có thể đã được clone với --single-branch.
    # Cho phép origin theo dõi mọi nhánh từ giờ trở đi.
    git("remote", "set-branches", "origin", "*")
    git("fetch", "origin")

elif project_path.exists() and any(project_path.iterdir()):
    raise RuntimeError(
        f"{project_path} đã tồn tại nhưng không phải Git repository. "
        "Hãy đổi PROJECT_DIR hoặc kiểm tra thư mục này."
    )

else:
    print("Đang clone repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(project_path)],
        check=True,
    )

# Báo lỗi rõ nếu tên nhánh không tồn tại trên origin.
remote_ref = f"refs/remotes/origin/{BRANCH}"
result = subprocess.run(
    ["git", "-C", str(project_path),
     "show-ref", "--verify", "--quiet", remote_ref],
)
if result.returncode != 0:
    raise RuntimeError(f"Không tìm thấy nhánh origin/{BRANCH}")

local_branch = subprocess.run(
    ["git", "-C", str(project_path),
     "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"],
)

if local_branch.returncode == 0:
    git("switch", BRANCH)
else:
    git("switch", "--track", f"origin/{BRANCH}")

git("pull", "--ff-only", "origin", BRANCH)

os.chdir(project_path)

actual_branch = subprocess.check_output(
    ["git", "-C", str(project_path), "branch", "--show-current"],
    text=True,
).strip()
commit = subprocess.check_output(
    ["git", "-C", str(project_path), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print(f"Project: {project_path}")
print(f"Branch:  {actual_branch}")
print(f"Commit:  {commit}")

Repository đã tồn tại, đang cập nhật...
Project: /content/AdvancedML
Branch:  main
Commit:  025f900


In [66]:
print("Nhánh thực tế:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())

Nhánh thực tế: main


In [67]:
# !git -C /content/AdvancedML remote set-branches --add origin feature/qa-qwen-streaming
# !git -C /content/AdvancedML fetch origin
# !git -C /content/AdvancedML switch --track origin/feature/qa-qwen-streaming
# !git -C /content/AdvancedML status --short --branch

# Cài Dependecy

In [68]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dtln.txt",
    ],
    check=True,
)

print("Dependencies đã sẵn sàng.")

Dependencies đã sẵn sàng.


In [69]:
# !apt-get update -qq && apt-get install -y ffmpeg

In [70]:
import shutil
print("FFmpeg:", shutil.which("ffmpeg"))

FFmpeg: /usr/bin/ffmpeg


# Kiểm tra GPU và thư viện

In [71]:
import gradio
import torch
import transformers

print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Gradio:       {gradio.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
else:
    print(
        "Cảnh báo: app đang dùng CPU. "
        "Hãy chọn Runtime → Change runtime type → GPU."
    )

PyTorch:      2.14.1+cu130
Transformers: 5.22.1
Gradio:       6.31.0
CUDA:         True
GPU:          NVIDIA L4


# Nạp checkpoint fine-tuned

In [72]:
from google.colab import drive
import os
import shutil
from pathlib import Path

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Đặt đường dẫn checkpoint trên Drive và trong project Colab
DRIVE_BEST_PT = Path("/content/drive/MyDrive/vvqa/checkpoints/separation/finetuned/best.pt")
TARGET_DIR = Path("/content/AdvancedML/checkpoints/separation/finetuned")
TARGET_FILE = TARGET_DIR / "best.pt"

TARGET_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_BEST_PT.exists():
    shutil.copy(DRIVE_BEST_PT, TARGET_FILE)
    print(f"Đã copy best.pt thành công! Dung lượng: {TARGET_FILE.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Không tìm thấy: {DRIVE_BEST_PT}")
    print("Vui lòng kiểm tra lại đường dẫn file trên Google Drive của bạn.")

Đã copy best.pt thành công! Dung lượng: 113.1 MB


In [73]:
# Copy checkpoint DTLN fine-tuned từ Drive
DRIVE_DTLN_H5 = Path("/content/drive/MyDrive/vvqa/checkpoints/enhancement/dtln/DTLN_vi_snr5_lr1e-4_run01.weights.h5")
TARGET_DTLN_DIR = Path("/content/AdvancedML/checkpoints/enhancement/dtln")
TARGET_DTLN_FILE = TARGET_DTLN_DIR / "DTLN_vi_snr5_lr1e-4_run01.weights.h5"

TARGET_DTLN_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_DTLN_H5.exists():
    shutil.copy(DRIVE_DTLN_H5, TARGET_DTLN_FILE)
    print(f"Đã copy DTLN checkpoint thành công! ({TARGET_DTLN_FILE.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Chưa có {DRIVE_DTLN_H5} trên Drive. DTLN sẽ tự động dùng pretrained model gốc.")

Đã copy DTLN checkpoint thành công! (11.9 MB)


In [74]:
# Trong configs/pipeline.yaml hoặc đặt biến môi trường:
import os
# Giữ GPU cho PyTorch, không để TensorFlow cấp phát độc quyền
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

# Chạy app

In [14]:
%cd /content/AdvancedML
!python -u -c "print('Python chạy được'); import app.main; print('Import app.main thành công')"

/content
Python chạy được
Import app.main thành công


In [15]:
# %cd /content/AdvancedML
# !python -u -m src.tts.infer --text "Thủ đô của Hàn Quốc là Seoul."

In [57]:
!git pull origin main

From https://github.com/thieuphong220203/vietnamese-voice-qa
 * branch            main       -> FETCH_HEAD
Already up to date.


In [75]:
%cd /content/AdvancedML
!python -u app/main.py --config configs/pipeline.yaml --host 0.0.0.0 --port 7860 --share

/content
Đang load pipeline từ: /content/AdvancedML/configs/pipeline.yaml
Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.
Loading model: vinai/PhoWhisper-medium
Device: cuda
Dtype: torch.float16
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 948/948 [00:00<00:00, 3134.38it/s]
PhoWhisper loaded successfully.
Loading QA model: Qwen/Qwen2.5-3B-Instruct
Device: cuda
Dtype: torch.float16
Loading weights: 100% 434/434 [00:00<00:00, 464.47it/s]
Pipeline đã sẵn sàng.
/content/AdvancedML/app/ui.py:318: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f674ebc46cce36a99f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permane